# Session 3 — Giving the Agent an Environment
### Tools, Tests, Git, and MCP

*Coding Agents That Improve* · Data Science Dojo × SambaNova · **Session 3 of 6**

---

**Session 1** split the agent in two: a frontier model plans, a fast SambaNova model executes.
**Session 2** turned one executor into many — and ended on a confession: our "executor" was
still just a model call. It had no hands.

A language model cannot edit a file. It cannot run a test. It emits text, and *something else*
has to turn that text into an action and report back what happened. That something else is the
**environment**, and it is most of what a coding agent actually is.

Today we build it:

| | |
|---|---|
| **Tools** | A typed contract the model can call — read, write, list, run |
| **A sandbox** | A real, isolated machine (Daytona) — because this code is written by a model |
| **Tests** | The feedback signal that closes the loop |
| **Git** | The deliverable — a reviewable diff, not a chat message |
| **MCP** | The universal protocol for everything you did not build yourself |

By the end, the agent takes a failing test suite and hands back a **PR-style diff that passes
CI** — plus a changelog entry it learned about from an MCP server.

## 0. Setup

You need two keys in `.env`:

* `SAMBANOVA_API_KEY` — [cloud.sambanova.ai](https://cloud.sambanova.ai/) (free tier)
* `DAYTONA_API_KEY` — [daytona.io](https://www.daytona.io/) (free tier, sign-up takes a minute)

In [1]:
# !pip install openai python-dotenv daytona mcp -q

In [2]:
import asyncio
import json
import os
import sys
import textwrap
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from daytona import Daytona, DaytonaConfig

load_dotenv()

# The executor. Swap freely — the harness is model-agnostic.
EXECUTOR = "MiniMax-M3"            # best tool-caller we measured: fewest steps, no malformed calls
# EXECUTOR = "gpt-oss-120b"        # ~2x faster and ~2.7x cheaper, slightly chattier

client = OpenAI(base_url="https://api.sambanova.ai/v1",
                api_key=os.environ["SAMBANOVA_API_KEY"])
daytona = Daytona(DaytonaConfig(api_key=os.environ["DAYTONA_API_KEY"]))

print(f"Executor model : {EXECUTOR}")
print(f"Sandboxes      : Daytona")

Executor model : MiniMax-M3
Sandboxes      : Daytona


---
## 1. Why a sandbox, and not your laptop

It is tempting to give the agent a `subprocess.run()` on your own machine. Three reasons not to:

1. **The code is written by a model.** Not malicious, usually — but `rm -rf` is one bad token
   away, and "clean up the build artefacts" is a plausible instruction to get wrong.
2. **Your laptop has your secrets.** Shell history, SSH keys, cloud credentials, other repos.
   A prompt injection in a dependency's README is a real exfiltration path.
3. **It does not scale.** Session 2's whole argument was N agents at once. N agents cannot share
   one working tree — that is a race condition, not a workspace.

A sandbox gives each run a disposable machine. The only thing that leaves is a diff.

In [3]:
t0 = time.perf_counter()
sandbox = daytona.create()
create_s = time.perf_counter() - t0

info = sandbox.process.exec("python3 --version; git --version; whoami; nproc")
print(f"sandbox {sandbox.id}")
print(f"created in {create_s:.1f}s\n")
print(info.result.strip())

sandbox 651fafab-16b8-4ee3-a3d6-90b9d68985ec
created in 1.3s

Python 3.14.4
git version 2.53.0
daytona
48


Seconds, not minutes. That number matters more than it looks: if a sandbox took two minutes to
boot, per-run isolation would be unaffordable and everyone would share one — which is exactly
how you get agents stepping on each other.

---
## 2. Seeding a real repo

We give the agent a small library with a genuine bug, and a test suite that catches it. This is
the **SWE-bench shape**: some tests fail now and must pass (`FAIL_TO_PASS`), and some already
pass and must keep passing (`PASS_TO_PASS`). The second half is what stops "fix" meaning
"delete the failing test".

In [4]:
REPO = "/home/daytona/project"

PRICING_PY = '''"""Tiny cart pricing library."""

def apply_discount(price, pct):
    """Apply a percentage discount to a price, rounded to 2dp."""
    return round(price - price * pct / 100, 2)


def cart_total(items, discount_pct=0):
    """Total a cart of (unit_price, qty) pairs, then apply a discount."""
    subtotal = 0
    for price, qty in items:
        subtotal += price * qty
    return apply_discount(subtotal, discount_pct)
'''

TEST_PY = '''import pytest
from pricing import apply_discount, cart_total

def test_apply_discount_basic():            # PASS_TO_PASS
    assert apply_discount(100, 10) == 90.0
    assert apply_discount(19.99, 0) == 19.99

def test_cart_total_basic():                # PASS_TO_PASS
    assert cart_total([(10.0, 2), (5.0, 1)]) == 25.0
    assert cart_total([(10.0, 2)], 50) == 10.0

def test_discount_must_be_in_range():       # FAIL_TO_PASS
    """A discount outside 0-100 is a bug, not a negative price."""
    with pytest.raises(ValueError):
        apply_discount(100, 150)
    with pytest.raises(ValueError):
        apply_discount(100, -5)

def test_cart_total_validates_discount():   # FAIL_TO_PASS
    with pytest.raises(ValueError):
        cart_total([(10.0, 1)], 200)
'''

def seed_repo(sb):
    """Upload the project and make a baseline git commit."""
    sb.process.exec(f"mkdir -p {REPO}")
    sb.fs.upload_file(PRICING_PY.encode(), f"{REPO}/pricing.py")
    sb.fs.upload_file(TEST_PY.encode(), f"{REPO}/test_pricing.py")
    sb.fs.upload_file(b"# Changelog\n\n## Unreleased\n", f"{REPO}/CHANGELOG.md")
    sb.fs.upload_file(b"__pycache__/\n*.pyc\n.pytest_cache/\n", f"{REPO}/.gitignore")
    for cmd in [
        "git init -q -b main",
        "git config user.email agent@sambanova.ai",
        "git config user.name 'SambaNova Coding Agent'",
        "git add -A",
        "git commit -q -m 'Initial cart pricing library'",
        "pip install -q pytest",
    ]:
        sb.process.exec(cmd, cwd=REPO)
    return sb.process.exec("git log --oneline", cwd=REPO).result.strip()

print(seed_repo(sandbox))

550499e Initial cart pricing library


### The baseline: red

In [5]:
baseline = sandbox.process.exec("python3 -m pytest -q --no-header -rf", cwd=REPO)
print(f"exit code: {baseline.exit_code}\n")
print(baseline.result[-1200:])

exit code: 1

..FF                                                                     [100%]
=================================== FAILURES ===================================
________________________ test_discount_must_be_in_range ________________________

    def test_discount_must_be_in_range():       # FAIL_TO_PASS
        """A discount outside 0-100 is a bug, not a negative price."""
>       with pytest.raises(ValueError):
             ^^^^^^^^^^^^^^^^^^^^^^^^^
E       Failed: DID NOT RAISE ValueError

test_pricing.py:14: Failed
______________________ test_cart_total_validates_discount ______________________

    def test_cart_total_validates_discount():   # FAIL_TO_PASS
>       with pytest.raises(ValueError):
             ^^^^^^^^^^^^^^^^^^^^^^^^^
E       Failed: DID NOT RAISE ValueError

test_pricing.py:20: Failed
=========================== short test summary info ============================
FAILED test_pricing.py::test_discount_must_be_in_range - Failed: DID NOT RAIS...
FAILED

Two failures, two passes. That is the agent's job description, and — importantly — its
**definition of done**. We never have to ask the model whether it succeeded.

---
## 3. The toolbelt

Now the contract. Each tool is a plain Python function over the sandbox, plus a JSON schema the
model sees. The schema *is* the API documentation as far as the model is concerned — vague
descriptions produce vague tool use.

In [6]:
import posixpath

def _rel(path):
    """Normalise a model-supplied path to repo-relative, or None if it escapes.

    Models hand back absolute paths as readily as relative ones. Without this,
    f"{REPO}/{path}" silently becomes /home/daytona/project/home/daytona/project/...
    — the write lands in a phantom file, the tests keep failing, and the agent
    has no way to tell. We hit exactly that; it cost a model a whole run.
    """
    p = (path or ".").strip()
    if p.startswith(REPO):
        p = p[len(REPO):]
    p = p.lstrip("/") or "."
    if posixpath.normpath(p).startswith(".."):
        return None
    return p


def tool_list_files(sb, path="."):
    rel = _rel(path)
    if rel is None:
        return "ERROR: path escapes the repository"
    return sb.process.exec(f"ls -la {rel}", cwd=REPO).result

def tool_read_file(sb, path, start=None, end=None):
    # Return the file VERBATIM. Tempting as line numbers are, a model holding a
    # full-file-overwrite tool will happily write them back into the source.
    rel = _rel(path)
    if rel is None:
        return "ERROR: path escapes the repository"
    out = sb.process.exec(f"cat {rel}", cwd=REPO).result or ""
    if start or end:
        lines = out.splitlines()
        out = "\n".join(lines[(int(start or 1) - 1):(int(end) if end else None)])
    return out

def tool_write_file(sb, path, content):
    rel = _rel(path)
    if rel is None:
        return "ERROR: path escapes the repository"
    sb.fs.upload_file(content.encode(), f"{REPO}/{rel}")
    return f"wrote {rel} ({len(content)} bytes)"

def tool_run_tests(sb):
    r = sb.process.exec("python3 -m pytest -q --no-header -rf", cwd=REPO)
    return f"exit_code={r.exit_code}\n{(r.result or '')[-2500:]}"

LOCAL_TOOLS = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the repository.",
        "parameters": {"type": "object",
                       "properties": {"path": {"type": "string", "description": "dir, default '.'"}},
                       "required": []}}},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file from the repo and return its exact contents.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"},
            "start": {"type": "integer", "description": "optional first line"},
            "end": {"type": "integer", "description": "optional last line"}},
            "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "write_file", "description": "Overwrite a file with new content (full contents).",
        "parameters": {"type": "object",
                       "properties": {"path": {"type": "string"}, "content": {"type": "string"}},
                       "required": ["path", "content"]}}},
    {"type": "function", "function": {
        "name": "run_tests", "description": "Run the pytest suite and return the output.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
]

LOCAL_DISPATCH = {"list_files": tool_list_files, "read_file": tool_read_file,
                  "write_file": tool_write_file, "run_tests": tool_run_tests}

print(f"{len(LOCAL_TOOLS)} tools defined")

4 tools defined


### The part every tutorial skips: the model will call your tools wrong

It will invent arguments you never declared, mistype a tool name, and pass a string where you
wanted an integer. If your dispatcher does `fn(**args)` the harness dies on a `TypeError` —
in the middle of a 40-turn run, after real work.

The fix is to treat a bad call as **data, not an exception**: validate against the schema and
hand the model a correctable error. Models recover from this well; they cannot recover from a
traceback in your process.

### The subtler one: paths

Argument *names* are the loud failure. Argument *values* are the quiet one. Some models return
`pricing.py`, others return `/home/daytona/project/pricing.py` — both entirely reasonable. If
your tool does `f"{REPO}/{path}"`, the absolute form silently becomes
`/home/daytona/project/home/daytona/project/pricing.py`: the write succeeds, lands in a phantom
file, and the tests keep failing with no indication why.

We hit this for real while building this session. One model used relative paths and sailed
through; another used absolute paths and burned every step of its budget before giving up. The
model was not worse — **the harness was wrong**, and only one model's habits exposed it. Hence
`_rel()` above, which normalises to repo-relative and refuses anything escaping the repo.

In [7]:
SCHEMA_ARGS = {t["function"]["name"]: set(t["function"]["parameters"]["properties"])
               for t in LOCAL_TOOLS}

def call_tool(sb, name, args, dispatch, schema_args):
    """Dispatch one tool call. Never raises — errors come back as text for the model."""
    if name not in dispatch:
        return f"ERROR: no such tool {name!r}. Available: {sorted(dispatch)}"
    unknown = set(args) - schema_args.get(name, set())
    if unknown:
        return (f"ERROR: unknown argument(s) {sorted(unknown)} for {name}. "
                f"Accepted: {sorted(schema_args[name])}")
    try:
        return dispatch[name](sb, **args)
    except Exception as e:
        return f"ERROR: {type(e).__name__}: {e}"

# Prove it: a hallucinated argument and a bogus tool name both survive.
print(call_tool(sandbox, "read_file", {"path": "pricing.py", "regex": "discount"},
                LOCAL_DISPATCH, SCHEMA_ARGS))
print(call_tool(sandbox, "grep_repo", {"q": "x"}, LOCAL_DISPATCH, SCHEMA_ARGS))

ERROR: unknown argument(s) ['regex'] for read_file. Accepted: ['end', 'path', 'start']
ERROR: no such tool 'grep_repo'. Available: ['list_files', 'read_file', 'run_tests', 'write_file']


---
## 4. The agent loop

This is the whole thing. Send messages and tool schemas, get back either text or tool calls,
execute them against the sandbox, append the results, repeat. Every coding agent you have used
is this loop with better prompts and more tools.

In [8]:
SYSTEM = f"""You are a coding agent working in a sandboxed git repo at {REPO}.

Fix the failing tests by editing the SOURCE code. Never edit the test file -
the tests define correct behaviour.

Workflow: read the failing test, read the source, make the smallest correct
change, then run the tests. Repeat until everything passes.
When the suite is green, reply with DONE and nothing else."""

def run_agent(sb, task, tools, dispatch, schema_args, max_steps=12, verbose=True):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": task}]
    steps = []
    for step in range(1, max_steps + 1):
        resp = client.chat.completions.create(
            model=EXECUTOR, messages=msgs, tools=tools, temperature=0.1, max_tokens=4000)
        m = resp.choices[0].message
        msgs.append(m.model_dump(exclude_none=True))

        if not m.tool_calls:
            if verbose:
                print(f"  {step:>2}. final: {(m.content or '').strip()[:100]}")
            steps.append({"step": step, "tool": None, "text": m.content})
            break

        for tc in m.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = call_tool(sb, name, args, dispatch, schema_args)
            shown = {k: (str(v)[:30] + "…" if len(str(v)) > 30 else v) for k, v in args.items()}
            first = str(result).strip().splitlines()[:1]
            if verbose:
                print(f"  {step:>2}. {name}({shown}) -> {first[0][:70] if first else ''}")
            steps.append({"step": step, "tool": name, "args": args})
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)[:6000]})
    return steps, msgs

In [9]:
t0 = time.perf_counter()
steps, _ = run_agent(sandbox, "The test suite is failing. Investigate and fix it.",
                     LOCAL_TOOLS, LOCAL_DISPATCH, SCHEMA_ARGS)
agent_s = time.perf_counter() - t0
print(f"\n{len(steps)} steps in {agent_s:.1f}s")

   1. list_files({'path': '.'}) -> total 20


   2. read_file({'path': 'pricing.py'}) -> """Tiny cart pricing library."""


   3. read_file({'path': 'test_pricing.py'}) -> import pytest


   4. run_tests({}) -> exit_code=1


   5. write_file({'content': '"""Tiny cart pricing library."…', 'path': 'pricing.py'}) -> wrote pricing.py (603 bytes)


   6. run_tests({}) -> exit_code=0


   7. final: DONE

7 steps in 17.0s


Watch the shape of that trace, not just the outcome. The agent **orients** (list, read), **acts**
(write), **checks** (run_tests) — and if the check fails, it goes back. Tests are what make that
loop terminate on something real rather than on the model deciding it feels finished.

### Which executor?

The harness is model-agnostic, so this is an empirical question rather than an opinion. Three
trials each on exactly this task, same tools, same prompt:

| | `MiniMax-M3` | `gpt-oss-120b` |
|---|---|---|
| Solved the task | 3/3 | 3/3 |
| Median steps | **6** | 8 |
| Median wall-clock | 11.0s | **6.1s** |
| Median cost | $0.0055 | **$0.0021** |
| Malformed tool calls | **0** | 2 |

`MiniMax-M3` is the better *agent* — it plans in fewer steps and never malformed a tool call.
`gpt-oss-120b` is roughly twice as fast and under half the price, and recovers fine from its own
mistakes. Which one you want depends on whether you are optimising the inner loop or the bill —
flip `EXECUTOR` at the top and re-run to see for yourself.

*(One task, three trials. Session 4 builds the harness that makes this kind of claim rigorous.)*

---
## 5. The CI gate

"It passes" is two claims: the broken thing works now, *and* nothing else broke. Check both.

In [10]:
final = sandbox.process.exec("python3 -m pytest -q --no-header -rf", cwd=REPO)
print(f"exit code: {final.exit_code}\n{final.result[-700:]}")

changed_files = sandbox.process.exec("git diff --name-only", cwd=REPO).result.strip().splitlines()
verdict = {
    "baseline_exit": baseline.exit_code,
    "final_exit": final.exit_code,
    "fail_to_pass_fixed": baseline.exit_code != 0 and final.exit_code == 0,
    "changed_files": changed_files,
    "tests_modified": any("test_" in f for f in changed_files),   # must be False
}
print("\n" + json.dumps(verdict, indent=2))

exit code: 0
....                                                                     [100%]
4 passed in 0.00s


{
  "baseline_exit": 1,
  "final_exit": 0,
  "fail_to_pass_fixed": true,
  "changed_files": [
    "pricing.py"
  ],
  "tests_modified": false
}


The `tests_modified` check is the one that matters for trust. If `test_pricing.py` appears in
the changed files, the agent moved the goalposts and the green suite means nothing. This is the
cheapest anti-gaming measure available, and it costs one `git diff --name-only`.

---
## 6. Git is the deliverable

The output of a coding agent is not a chat message. It is a **diff a human can review** — which
is also why git belongs in the toolbelt rather than in your orchestration code.

In [11]:
diff = sandbox.process.exec("git diff", cwd=REPO).result
print(diff)

diff --git a/pricing.py b/pricing.py
index a6ece06..32e7f9b 100644
--- a/pricing.py
+++ b/pricing.py
@@ -1,7 +1,13 @@
 """Tiny cart pricing library."""
 
+
 def apply_discount(price, pct):
-    """Apply a percentage discount to a price, rounded to 2dp."""
+    """Apply a percentage discount to a price, rounded to 2dp.
+
+    Raises ValueError if pct is not in the range [0, 100].
+    """
+    if pct < 0 or pct > 100:
+        raise ValueError(f"discount percentage must be between 0 and 100, got {pct}")
     return round(price - price * pct / 100, 2)
 
 



In [12]:
for cmd in ["git checkout -q -b fix/cart-412",
            "git add -A",
            "git commit -q -m 'Validate discount percentage range (CART-412)'"]:
    sandbox.process.exec(cmd, cwd=REPO)

print(sandbox.process.exec("git log --oneline -2; git branch --show-current", cwd=REPO).result)
print("\n--- PR-style patch ---")
print(sandbox.process.exec("git format-patch -1 --stdout | head -40", cwd=REPO).result)

8cb236f Validate discount percentage range (CART-412)
550499e Initial cart pricing library
fix/cart-412


--- PR-style patch ---
From 8cb236f850b2f5b52da1e05e1ec40f4745f74302 Mon Sep 17 00:00:00 2001
From: SambaNova Coding Agent <agent@sambanova.ai>
Date: Wed, 21 Oct 2026 17:25:47 +0000
Subject: [PATCH] Validate discount percentage range (CART-412)

---
 pricing.py | 8 +++++++-
 1 file changed, 7 insertions(+), 1 deletion(-)

diff --git a/pricing.py b/pricing.py
index a6ece06..32e7f9b 100644
--- a/pricing.py
+++ b/pricing.py
@@ -1,7 +1,13 @@
 """Tiny cart pricing library."""
 
+
 def apply_discount(price, pct):
-    """Apply a percentage discount to a price, rounded to 2dp."""
+    """Apply a percentage discount to a price, rounded to 2dp.
+
+    Raises ValueError if pct is not in the range [0, 100].
+    """
+    if pct < 0 or pct > 100:
+        raise ValueError(f"discount percentage must be between 0 and 100, got {pct}")
     return round(price - price * pct / 100, 2)
 
 
-- 
2.53.0

---
## 7. MCP — the tools you did not write

Our four tools are hand-rolled. Real agents need dozens: issue trackers, docs search, CI status,
observability, internal services. Writing and maintaining a bespoke adapter for each, per agent
framework, is the integration problem that ate everyone's Q3.

**MCP (Model Context Protocol)** is the answer that stuck: servers expose tools over a standard
protocol, and any client — Claude Code, Cursor, Codex, or the loop we just wrote — can discover
and call them. Write the server once; every agent can use it.

Here is a real one, in about twenty lines.

In [13]:
MCP_SERVER = textwrap.dedent('''
    """A tiny MCP server exposing repo intelligence: issues and house rules."""
    from mcp.server.fastmcp import FastMCP

    mcp = FastMCP("repo-intel")

    @mcp.tool()
    def changelog_policy() -> str:
        """Return this project's house rule for changelog entries."""
        return ("House rule: every behaviour change needs a CHANGELOG.md entry under "
                "'## Unreleased', written as '- <verb> <what> (<ISSUE-ID>)'.")

    @mcp.tool()
    def lookup_issue(issue_id: str) -> str:
        """Look up an issue by id and return its title and acceptance criteria."""
        issues = {
            "CART-412": {
                "title": "apply_discount accepts out-of-range percentages",
                "criteria": ("Reject pct < 0 or pct > 100 with ValueError. "
                             "cart_total must validate too."),
            }
        }
        i = issues.get(issue_id.upper())
        return (f"{issue_id}: {i['title']}\\nAcceptance criteria: {i['criteria']}"
                if i else f"No issue {issue_id}")

    if __name__ == "__main__":
        mcp.run()
''').strip()

Path("mcp_repo_intel.py").write_text(MCP_SERVER)
print("wrote mcp_repo_intel.py")

wrote mcp_repo_intel.py


### Discovering its tools over the protocol

Note what we *don't* do: import it, or know anything about it beyond how to start it. We ask it
what it can do, and it tells us — schemas included. That is the whole point.

In [14]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PARAMS = StdioServerParameters(
    command=sys.executable, args=["mcp_repo_intel.py"], env=dict(os.environ))

async def mcp_list_tools():
    async with stdio_client(SERVER_PARAMS) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            return [(t.name, t.description, t.inputSchema) for t in (await s.list_tools()).tools]

mcp_tools = await mcp_list_tools()
for name, desc, schema in mcp_tools:
    print(f"  {name}({', '.join(schema.get('properties', {}))}) — {desc}")

  changelog_policy() — Return this project's house rule for changelog entries.
  lookup_issue(issue_id) — Look up an issue by id and return its title and acceptance criteria.


### Bridging MCP tools into the agent

An MCP tool definition and an OpenAI function definition are the same three things: a name, a
description, and a JSON schema. So the bridge is mechanical — and it means *any* MCP server in
the ecosystem becomes available to the agent we built, with no new adapter code.

In [15]:
def mcp_to_openai(name, description, schema):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object",
                       "properties": schema.get("properties", {}),
                       "required": schema.get("required", [])}}}

async def mcp_call(name, args):
    async with stdio_client(SERVER_PARAMS) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            res = await s.call_tool(name, args)
            return "\n".join(getattr(c, "text", str(c)) for c in res.content)

MCP_SCHEMAS = {n: s for n, _, s in mcp_tools}

def make_mcp_dispatch(name):
    # The agent loop is sync; MCP is async. Bridge with a nested event loop per call.
    def _fn(sb, **kwargs):
        return asyncio.get_event_loop().run_until_complete(mcp_call(name, kwargs))
    return _fn

print(json.dumps(mcp_to_openai(*mcp_tools[1]), indent=2)[:420])

{
  "type": "function",
  "function": {
    "name": "lookup_issue",
    "description": "Look up an issue by id and return its title and acceptance criteria.",
    "parameters": {
      "type": "object",
      "properties": {
        "issue_id": {
          "title": "Issue Id",
          "type": "string"
        }
      },
      "required": [
        "issue_id"
      ]
    }
  }
}


### The agent with a bigger world

Same loop, same sandbox — a wider toolbelt. Now ask for something the agent **cannot** know from
the repo alone: the house changelog rule lives in someone else's system.

In [16]:
import nest_asyncio
nest_asyncio.apply()          # lets the sync agent loop drive async MCP calls inside Jupyter

ALL_TOOLS = LOCAL_TOOLS + [mcp_to_openai(n, d, s) for n, d, s in mcp_tools]
ALL_DISPATCH = {**LOCAL_DISPATCH, **{n: make_mcp_dispatch(n) for n, _, _ in mcp_tools}}
ALL_SCHEMA_ARGS = {**SCHEMA_ARGS,
                   **{n: set(s.get("properties", {})) for n, s in MCP_SCHEMAS.items()}}

print(f"toolbelt: {len(ALL_TOOLS)} tools "
      f"({len(LOCAL_TOOLS)} local + {len(mcp_tools)} over MCP)\n")

steps2, _ = run_agent(
    sandbox,
    "Issue CART-412 has been filed against this repo. Look it up and confirm the current code "
    "satisfies its acceptance criteria. Then add a CHANGELOG.md entry for it, following our "
    "project's changelog house rule - use your tools to find out what that rule is, do not "
    "guess it. Edit CHANGELOG.md only; the source is already correct.",
    ALL_TOOLS, ALL_DISPATCH, ALL_SCHEMA_ARGS, max_steps=18)

toolbelt: 6 tools (4 local + 2 over MCP)



   1. lookup_issue({'issue_id': 'CART-412'}) -> CART-412: apply_discount accepts out-of-range percentages


   1. changelog_policy({}) -> House rule: every behaviour change needs a CHANGELOG.md entry under '#
   1. list_files({}) -> total 20


   2. read_file({'path': 'pricing.py'}) -> """Tiny cart pricing library."""
   2. read_file({'path': 'test_pricing.py'}) -> import pytest


   2. read_file({'path': 'CHANGELOG.md'}) -> # Changelog


   3. run_tests({}) -> exit_code=0


   4. write_file({'content': '# Changelog\n\n## Unreleased\n- R…', 'path': 'CHANGELOG.md'}) -> wrote CHANGELOG.md (114 bytes)


   5. read_file({'path': 'CHANGELOG.md'}) -> # Changelog


   5. run_tests({}) -> exit_code=0


   6. final: DONE


In [17]:
print(sandbox.process.exec("cat CHANGELOG.md", cwd=REPO).result)
print("--- final suite ---")
r = sandbox.process.exec("python3 -m pytest -q --no-header", cwd=REPO)
print(f"exit code: {r.exit_code}\n{r.result[-300:]}")

# Changelog

## Unreleased
- Reject out-of-range discount percentages in apply_discount and cart_total (CART-412)

--- final suite ---


exit code: 0
....                                                                     [100%]
4 passed in 0.00s



The agent did not know the changelog convention, and it did not invent one. It **asked a server**
— the same way it would ask Jira, GitHub, or your internal docs in production. That is the
difference MCP makes: the agent's capabilities stop being a function of what you hard-coded.

---
## 8. Operating notes

What changes when this leaves a notebook:

* **Scope tools per agent.** A reviewer agent needs `read_file`, not `write_file`. The toolbelt
  is your permission model — the narrowest one that does the job.
* **Timeouts everywhere.** `run_tests` on a real repo can hang forever. Bound the command, the
  step count, and the run.
* **Tool errors are normal.** We validated arguments; also handle a failing command, a missing
  file, and a test suite that errors instead of failing. Each should return text, not raise.
* **The sandbox has a lifecycle.** Create per session, reuse across turns (clone + install is
  the big fixed cost), reap when idle, and always delete in a `finally` — leaked sandboxes are
  how a fleet quietly gets expensive.
* **Keep credentials out of the sandbox.** It runs model-written code. Proxy git and model
  traffic, inject real credentials server-side, and allowlist egress.
* **MCP servers are dependencies.** They are processes you spawn and trust with context. Pin
  them, scope them per agent, and read what a third-party server actually does before attaching it.

In [18]:
# Always clean up. In production this belongs in a `finally`.
sandbox.delete()
print(f"sandbox deleted")

sandbox deleted


---
## Takeaways

1. **A model is not an agent.** The environment — tools, sandbox, tests, git — is most of what a
   coding agent is, and all of what makes it trustworthy.
2. **The sandbox is not optional.** The code is model-written, your laptop has your secrets, and
   N agents cannot share one working tree. Per-run isolation only works because creating one
   takes seconds.
3. **Tests are the loop's terminating condition.** Without them the agent stops when it feels
   done; with them it stops when it *is* done — and `git diff --name-only` stops it from cheating.
4. **Your dispatcher must not raise.** The model will call tools wrong. Hand it a correctable
   error, not a traceback.
5. **Git is the deliverable.** A reviewable diff on a branch, not a chat message.
6. **MCP is how the toolbelt grows** past what you wrote yourself — one server, every client.

### Next session
**Session 4 — Evaluating Coding Agents: Tests as Ground Truth, Trajectories, and SambaEval.**
We built the environment; now we measure whether the agent uses it well — pass@k over a task
set, trajectory evaluation, and picking the executor model on evidence.